# GPU Architecture and a First CUDA Program

This notebook contains **one complete CUDA example**: adding two vectors,
$c_i=a_i+b_i$. Python notebook cells prepare files and launch the compiler;
the numerical program itself is written in CUDA C++.

By the end, you will have written a kernel, allocated device arrays,
transferred inputs, launched GPU threads, copied the output back,
printed the sum, validated every value against a CPU reference, and
released device memory. This notebook measures correctness, not speed.

In Google Colab select **Runtime → Change runtime type → GPU** before
running the cells in order. A GPU, compatible driver, CUDA compiler (`nvcc`),
and host C++ compiler must be available in the runtime. No additional
CUDA notebook extension is used.

## 1. Inspect the runtime

The exclamation mark runs a command in the remote Linux shell.
`nvidia-smi` reports the assigned GPU and driver support;
`nvcc --version` reports the CUDA compiler version. These version
numbers describe different components and need not be identical.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
from pathlib import Path
import shutil
import subprocess

for command in ("nvidia-smi", "nvcc", "g++"):
    if shutil.which(command) is None:
        raise RuntimeError(
            f"{command} is unavailable. Select a GPU runtime and check its CUDA toolchain."
        )
subprocess.run(["nvidia-smi", "-L"], check=True)
Path("code").mkdir(exist_ok=True)

## 2. Write the kernel header

`%%writefile` saves the following text in a file. It does not compile it.
Each thread computes at most one element. The bounds check protects array
accesses when the grid contains more threads than valid elements.

`__global__` identifies a GPU kernel. `extern "C"` preserves its unmangled
name for later Python-interface examples; this C++ host program would
also work without that declaration.

In [ ]:
%%writefile code/chapter02_vector_add_kernel.cuh
#pragma once

extern "C" __global__ void vector_add(const float *a,
                                     const float *b,
                                     float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}


## 3. Write the host program

The program creates deterministic single-precision input arrays.
It explicitly allocates three device arrays, transfers the two inputs,
launches the kernel, checks errors, waits, and copies the output back.
`CUDA_CHECK` is a helper defined in this example, not a CUDA API function.

Every output is compared with `a[i] + b[i]` computed on the CPU. The test
rejects nonfinite results and uses an absolute-plus-relative tolerance.
These particular inputs are binary fractions, so exact agreement is
expected, but the validation does not assume exact floating-point
agreement for arbitrary future inputs.

The array length is an optional command-line argument (default: 19).
The block size remains 256. Allocations are freed before the program ends.

In [ ]:
%%writefile code/chapter02_vector_add_cuda.cu
#include <cuda_runtime.h>
#include <algorithm>
#include <cerrno>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <limits>
#include <vector>
#include "chapter02_vector_add_kernel.cuh"

#define CUDA_CHECK(call) do {                                      \
    cudaError_t status = (call);                                   \
    if (status != cudaSuccess) {                                  \
        std::fprintf(stderr, "%s:%d: %s failed: %s\n",              \
                     __FILE__, __LINE__, #call,                    \
                     cudaGetErrorString(status));                 \
        std::exit(EXIT_FAILURE);                                  \
    }                                                             \
} while (0)

int main(int argc, char **argv)
{
    int n = 19;
    if (argc > 2) {
        std::fprintf(stderr, "Usage: %s [positive_length]\n", argv[0]);
        return EXIT_FAILURE;
    }
    if (argc == 2) {
        errno = 0;
        char *end = nullptr;
        const long value = std::strtol(argv[1], &end, 10);
        if (errno != 0 || end == argv[1] || *end != '\0' ||
            value < 1 || value > 1000000) {
            std::fprintf(stderr, "Choose a length from 1 to 1000000.\n");
            return EXIT_FAILURE;
        }
        n = static_cast<int>(value);
    }

    const size_t bytes = size_t(n) * sizeof(float);
    std::vector<float> a(n), b(n);
    std::vector<float> c(n, std::numeric_limits<float>::quiet_NaN());
    for (int i = 0; i < n; ++i) {
        a[i] = 0.5f * float(i);
        b[i] = 1.0f + 0.25f * float(i % 7);
    }

    float *d_a = nullptr;
    float *d_b = nullptr;
    float *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, bytes, cudaMemcpyDeviceToHost));

    int failures = 0;
    double max_error = 0.0;
    for (int i = 0; i < n; ++i) {
        const float expected = a[i] + b[i];
        const double error = std::fabs(double(c[i]) - double(expected));
        const double tolerance = 1.0e-6 + 1.0e-6 * std::fabs(double(expected));
        if (!std::isfinite(c[i]) || error > tolerance) {
            ++failures;
        }
        max_error = std::isfinite(error)
                  ? std::max(max_error, error)
                  : std::numeric_limits<double>::infinity();
    }

    std::printf("N = %d, threads per block = %d, blocks = %d\n",
                n, threads, blocks);
    std::printf("Threads without a valid element: %d\n\n",
                blocks * threads - n);
    std::printf("     i          a[i]          b[i]       GPU c[i]       CPU a+b\n");
    for (int i = 0; i < n; ++i) {
        // Print every element for small arrays; head and tail otherwise.
        if (n <= 20 || i < 8 || i >= n - 4) {
            std::printf("%6d %13.4f %13.4f %14.4f %13.4f\n",
                        i, double(a[i]), double(b[i]),
                        double(c[i]), double(a[i] + b[i]));
        } else if (i == 8) {
            std::printf("   ... (remaining elements are also validated) ...\n");
        }
    }
    std::printf("\nMaximum absolute error: %.6g\n", max_error);
    std::printf("Validation: %s (%d failures among %d elements)\n",
                failures == 0 ? "PASS" : "FAIL", failures, n);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return failures == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}


## 4. Compile

This cell runs a Bash script. `-O2` requests host optimization level 2,
`-std=c++14` selects C++14, and `-o` names the executable.
If supported, `-arch=native` targets the GPU assigned to this runtime.
Recompile when starting a new session or changing the assigned GPU.

The old executable is removed before compiling, so a failed build cannot
silently leave an older program to run. The cell stops on compilation failure.

In [ ]:
%%bash
set -e
rm -f vector_add_cuda
if nvcc --help | grep -q "native"; then
    nvcc -O2 -std=c++14 -arch=native code/chapter02_vector_add_cuda.cu -o vector_add_cuda
else
    nvcc -O2 -std=c++14 code/chapter02_vector_add_cuda.cu -o vector_add_cuda
fi

## 5. Run and display the vector sum

This cell executes the compiled program and displays its output directly.
For 19 elements it prints every input pair and every GPU sum, together
with the CPU reference. Check the first entry yourself: $0+1=1$.
The cell raises an error if the executable reports a CUDA or validation failure.

In [ ]:
subprocess.run(["./vector_add_cuda", "19"], check=True);

## 6. Check the same program across a block boundary

With 259 elements and 256 threads per block, the grid has two blocks.
It launches 512 threads, of which 253 have no valid element. Only the
first eight and last four rows are printed, but **all 259 outputs are
validated**, including the final elements. This is another input to the
same vector-addition program, not a second algorithm or a timing experiment.

In [ ]:
subprocess.run(["./vector_add_cuda", "259"], check=True);

## Exercises

1. With 1000 elements and 256 threads per block, how many blocks and
   out-of-range threads are created? What index belongs to thread 17 of block 2?
2. Assuming four-byte floats, how much device storage and transfer volume
   does the 1000-element case require? Does synchronization copy the output?
3. What does `cudaMalloc` return? Are its values initialized? Can a pointer
   cast convert double-precision data into single-precision data?
4. Run the same executable with lengths 1, 255, 256, and 257. Predict the
   launch sizes and verify that every case passes.
5. Modify the kernel to compute y[i] = 2 x[i] + y[i] in place (SAXPY).
   Which arrays must be copied to the GPU, and which back to the host?
6. Change `threads` from 256 to 2048, recompile, and run. Which call reports
   the problem, and why does the kernel never run?
7. Remove the test `i < n` from the kernel and run the program with 257
   elements under `!compute-sanitizer ./vector_add_cuda 257`. What does the
   tool report? Why might the program appear correct without it?

The solutions are collected at the end of the book.

In [ ]:
# Optional validation exercise: rerun the same compiled program.
for length in (1, 255, 256, 257):
    print(f"\n--- Validation case: N={length} ---", flush=True)
    subprocess.run(["./vector_add_cuda", str(length)], check=True)

## What you have completed

You have executed and validated a complete CUDA program, from host input
to printed GPU output. The next chapter changes the organization of the
work and introduces performance measurements.

Save the notebook in Google Drive to retain its cells. The generated
source files and executable live in the temporary runtime and can be
recreated by rerunning the notebook. No figures, spreadsheets, or result
files are exported automatically.

## References

- NVIDIA, [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/index.html).
- NVIDIA, [CUDA Runtime API Reference Manual](https://docs.nvidia.com/cuda/cuda-runtime-api/index.html).
- NVIDIA, [CUDA Compiler Driver NVCC](https://docs.nvidia.com/cuda/cuda-compiler-driver-nvcc/index.html).
- Google, [Colab FAQ](https://research.google.com/colaboratory/faq.html).
